# Syed Huzaifa Rahman 24I-2500 DS-A

## Assignment 2 Game UNO

### Players
### Player 1 (AI Minimax Defensive)
### Player 2 (AI / Expectimax Offensive)
### Player 3 (User /AI Manual or Simulation)

In [1]:
# 1 push 
import numpy as np
import random

colors = ["Red", "Blue", "Green", "Yellow"]
skip = "skip"

class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value
        self._validate()

    def _validate(self):
        if type(self.color) is not str:
            raise TypeError("Card color must be a string")
        if self.color not in colors:
            raise ValueError("Invalid color: " + str(self.color))

        if type(self.value) is int:
            if self.value < 0 or self.value > 9:
                raise ValueError("Card number must be 0 to 9")
        elif type(self.value) is str:
            if self.value != skip:
                raise ValueError("Invalid special value: " + str(self.value))
        else:
            raise TypeError("Card value must be int or 'skip'")

    def is_skip(self):
        return type(self.value) is str and self.value == skip

    def matches(self, top_card):
        # same color or number
        if self.color == top_card.color:
            return True
        if self.is_skip() or top_card.is_skip():
            return False
        return self.value == top_card.value

    def __str__(self):
        return self.color + " " + str(self.value)


class GameState:
    def __init__(self, hands, top_card, deck, current_player):
        self.hands = hands
        self.top_card = top_card
        self.deck = deck
        # 0,1,2
        self.current_player = current_player

    def clone(self):
        new_hands = []
        i = 0
        while i < 3:
            new_hands.append(list(self.hands[i]))
            i += 1
        return GameState(new_hands, self.top_card, list(self.deck), self.current_player)

    def is_terminal(self):
        i = 0
        while i < 3:
            if len(self.hands[i]) == 0:
                return True
            i += 1
        return False


def generate_deck():
    # deck has 0 to 9 of every color and a skip then shiffled
    deck = []
    i = 0
    while i < len(colors):
        c = colors[i]
        v = 0
        while v <= 9:
            deck.append(Card(c, v))
            v += 1
        deck.append(Card(c, skip))
        i += 1

    # shuffle using constraint
    perm = np.random.permutation(len(deck))
    shuffled = []
    k = 0
    while k < len(perm):
        shuffled.append(deck[int(perm[k])])
        k += 1
    return shuffled


def _take_last(deck):
    # pick/draw card
    card = deck[len(deck) - 1]
    del deck[len(deck) - 1]
    return card


def deal_initial_state():
    # every player gets 5 non skip cards
    deck = generate_deck()
    hands = [[], [], []]

    r = 0
    while r < 5:
        p = 0
        while p < 3:
            hands[p].append(_take_last(deck))
            p += 1
        r += 1

    while True:
        top = _take_last(deck)
        if not top.is_skip():
            top_card = top
            break
        deck.insert(0, top)

    return GameState(hands, top_card, deck, 0)


# checking random cards and shuffle here (state change every time cell runs)
_state0 = deal_initial_state()
print("Top card:", str(_state0.top_card))
print("Hand sizes:", [len(h) for h in _state0.hands], "Deck:", len(_state0.deck))

Top card: Red 5
Hand sizes: [5, 5, 5] Deck: 28


In [3]:
# Legal moves and State transition
# push 1
def get_valid_moves(hand, top_card):
    plays = []
    i = 0
    while i < len(hand):
        if hand[i].matches(top_card):
            plays.append(("play", hand[i]))
        i += 1
    if len(plays) > 0:
        return plays
        
    return [("draw", None)]

def _remove_first_occurrence(hand, card):
    i = 0
    while i < len(hand):
        if hand[i].color == card.color and hand[i].value == card.value:
            del hand[i]
            return True
        i += 1
    return False

def apply_move(state, move):
    new_state = state.clone()
    player = new_state.current_player
    hand = new_state.hands[player]
    action = move[0]
    card = move[1]
    if action == "play":
        if card is None:
            raise ValueError("Play move requires a card")
            
        if not card.matches(new_state.top_card):
            raise ValueError("Illegal move")

        ok = _remove_first_occurrence(hand, card)
        if not ok:
            raise ValueError("Card not found in hand")

        new_state.top_card = card
        # skip skips next player's turn
        if card.is_skip():
            new_state.current_player = (player + 2) % 3
        else:
            new_state.current_player = (player + 1) % 3
        return new_state

    if action == "draw":
        # Rule 2 draw exactly 1 if no valid move
        if len(new_state.deck) > 0:
            hand.append(_take_last(new_state.deck))
            
        new_state.current_player = (player + 1) % 3
        return new_state
    raise ValueError("Invalid move action")

# quick check to see for every time program runs 
hand = _state0.hands[_state0.current_player]
moves = get_valid_moves(hand, _state0.top_card)
print("Current player:", _state0.current_player)
print("Valid moves:", [m[0] if m[0] == "draw" else str(m[1]) for m in moves])

Current player: 0
Valid moves: ['Green 5', 'Yellow 5']


In [4]:
# Eval funcs defensive/offensive
# push 2
def eval_base(state: GameState, player_index: int) -> float:
    c_ai = float(len(state.hands[player_index]))
    opp_cards = []
    for i in range(3):
        if i != player_index:
            opp_cards.append(len(state.hands[i]))
    c_opp = float(sum(opp_cards)) / 2.0
    s = 0.0
    for card in state.hands[player_index]:
        if card.is_skip():
            s += 1.0
    return 50.0 - 5.0 * c_ai + 2.0 * c_opp + 3.0 * s

def eval_defensive(state: GameState, player_index: int) -> float:
    # we prioritize blocking opps that are close to winning
    score = eval_base(state, player_index)
    for i in range(3):
        if i == player_index:
            continue
        c = len(state.hands[i])
        if c <= 2:
            # extra penalty if opps are near 0
            score -= 8.0 * float(3 - c)
    return score

def eval_offensive(state: GameState, player_index: int) -> float:
    # prioritize remove cards more
    c_ai = float(len(state.hands[player_index]))
    opp_cards = []
    for i in range(3):
        if i != player_index:
            opp_cards.append(len(state.hands[i]))
    c_opp = float(sum(opp_cards)) / 2.0
    s = 0.0
    for card in state.hands[player_index]:
        if card.is_skip():
            s += 1.0
    return 50.0 - 7.0 * c_ai + 1.0 * c_opp + 4.0 * s

print("Baseline score for P1:", eval_base(_state0, 0))
print("Defensive score for P1:", eval_defensive(_state0, 0))
print("Offensive score for P2:", eval_offensive(_state0, 1))

Baseline score for P1: 35.0
Defensive score for P1: 35.0
Offensive score for P2: 24.0


In [5]:
# minimax depth =3 and expectimax depth=3

# push 3

def minimax(state, depth, max_player, eval_fn):
    if depth == 0 or state.is_terminal():
        return float(eval_fn(state, max_player)), None
    current = state.current_player
    moves = get_valid_moves(state.hands[current], state.top_card)

    if current == max_player:
        best_val = -10000
        best_move = None
        i = 0
        while i < len(moves):
            child = apply_move(state, moves[i])
            val, _ = minimax(child, depth - 1, max_player, eval_fn)
            if val > best_val:
                best_val = val
                best_move = moves[i]
            i += 1
        return best_val, best_move

    best_val = 10000
    best_move = None
    i = 0
    while i < len(moves):
        child = apply_move(state, moves[i])
        val, _ = minimax(child, depth - 1, max_player, eval_fn)
        if val < best_val:
            best_val = val
            best_move = moves[i]
        i += 1
    return best_val, best_move


def _remove_deck_index(deck, idx):
    # remove value at deck[idx]
    card = deck[idx]
    del deck[idx]
    return card


def expectimax(state, depth, max_player, eval_fn):
    if depth == 0 or state.is_terminal():
        return float(eval_fn(state, max_player))
    current = state.current_player
    moves = get_valid_moves(state.hands[current], state.top_card)
    # max node
    if current == max_player:
        best_val = -10000
        i = 0
        while i < len(moves):
            mv = moves[i]
            if mv[0] == "draw":
                val = expectimax_chance_node(state, depth, max_player, eval_fn)
            else:
                child = apply_move(state, mv)
                val = expectimax(child, depth - 1, max_player, eval_fn)

            if val > best_val:
                best_val = val
            i += 1
        return best_val

    # opp node which is a random legal move
    mv = random.choice(moves)
    if mv[0] == "draw":
        return expectimax_chance_node(state, depth, max_player, eval_fn)
    child = apply_move(state, mv)
    return expectimax(child, depth - 1, max_player, eval_fn)


def expectimax_chance_node(state, depth, max_player, eval_fn):
    # chance node where you draw one card randomly from deck
    if depth == 0 or state.is_terminal():
        return float(eval_fn(state, max_player))
    if len(state.deck) == 0:
        return float(eval_fn(state, max_player))

    total = float(len(state.deck))
    expected = 0.0
    i = 0
    while i < len(state.deck):
        child = state.clone()
        player = child.current_player
        drawn = _remove_deck_index(child.deck, i)
        child.hands[player].append(drawn)
        child.current_player = (player + 1) % 3
        val = expectimax(child, depth - 1, max_player, eval_fn)
        expected += (1.0 / total) * val
        i += 1
    return expected


def choose_move_minimax(state, player_index):
    _, mv = minimax(state, 3, player_index, eval_defensive)
    if mv is None:
        return get_valid_moves(state.hands[player_index], state.top_card)[0]
    return mv


def choose_move_expectimax(state, player_index):
    moves = get_valid_moves(state.hands[player_index], state.top_card)
    best_mv = moves[0]
    best_val = -10000
    i = 0
    while i < len(moves):
        mv = moves[i]
        if mv[0] == "draw":
            val = expectimax_chance_node(state, 3, player_index, eval_offensive)
        else:
            child = apply_move(state, mv)
            val = expectimax(child, 2, player_index, eval_offensive)
        if val > best_val:
            best_val = val
            best_mv = mv
        i += 1
    return best_mv


print("Minimax suggested move for P1:", choose_move_minimax(_state0, 0))

# quick check to see expectimax for P2's turn
_tmp = _state0.clone()
_tmp.current_player = 1
print("Expectimax suggested move for P2:", choose_move_expectimax(_tmp, 1))

Minimax suggested move for P1: ('play', <__main__.Card object at 0x12fcefb90>)
Expectimax suggested move for P2: ('play', <__main__.Card object at 0x12fcd4ec0>)
